In [1]:
# Run once if packages are missing
%pip install -q numpy pandas librosa opencv-python tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import logging
import math
import os
import shutil
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable

import cv2
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
LOGGER = logging.getLogger('gunshot_pipeline')

d:\anaconda3\envs\LLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## config

In [3]:
@dataclass
class PipelineConfig:
    # Required paths
    dataset_root: Path
    output_root: Path

    # File discovery (added '.wav')
    video_extensions: tuple[str, ...] = ('.mp4', '.mov', '.mkv', '.avi', '.webm', '.wav')

    # Audio extraction
    sample_rate: int = 22050

    # Audio event detection
    frame_length: int = 2048
    hop_length: int = 256
    zscore_threshold: float = 3.0
    min_event_gap_s: float = 0.20

    # Frequency focus for impulsive events (gunshot-like)
    low_band_hz: float = 150.0
    high_band_hz: float = 3500.0

    # Visual validation around candidate timestamp
    visual_window_s: float = 0.35
    flash_zscore_threshold: float = 2.5
    motion_zscore_threshold: float = 2.0

    # Trimming behavior
    pre_event_pad_s: float = 0.2
    post_event_pad_s: float = 0.5
    merge_overlap_s: float = 0.10

    # Runtime
    overwrite: bool = True

CONFIG = PipelineConfig(
    dataset_root=Path(r'c:\Desktop\Data-Cleaner\research\Data'),
    output_root=Path(r'c:\Desktop\Data-Cleaner\research\Data\Output'),
)

CONFIG.output_root.mkdir(parents=True, exist_ok=True)
print('Config ready:')
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(CONFIG).items()}, indent=2))


Config ready:
{
  "dataset_root": "c:\\Desktop\\Data-Cleaner\\research\\Data",
  "output_root": "c:\\Desktop\\Data-Cleaner\\research\\Data\\Output",
  "video_extensions": [
    ".mp4",
    ".mov",
    ".mkv",
    ".avi",
    ".webm",
    ".wav"
  ],
  "sample_rate": 22050,
  "frame_length": 2048,
  "hop_length": 256,
  "zscore_threshold": 3.0,
  "min_event_gap_s": 0.2,
  "low_band_hz": 150.0,
  "high_band_hz": 3500.0,
  "visual_window_s": 0.35,
  "flash_zscore_threshold": 2.5,
  "motion_zscore_threshold": 2.0,
  "pre_event_pad_s": 0.2,
  "post_event_pad_s": 0.5,
  "merge_overlap_s": 0.1,
  "overwrite": true
}


## utility

In [4]:
def require_ffmpeg() -> None:
    if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
        raise EnvironmentError('ffmpeg and ffprobe are required in PATH.')


def run_cmd(cmd: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, check=True, text=True, capture_output=True)


def get_video_duration_s(video_path: Path) -> float:
    cmd = [
        'ffprobe',
        '-v',
        'error',
        '-show_entries',
        'format=duration',
        '-of',
        'default=noprint_wrappers=1:nokey=1',
        str(video_path),
    ]
    out = run_cmd(cmd).stdout.strip()
    return float(out)


def robust_zscore(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + 1e-8
    return 0.6745 * (x - med) / mad


def discover_videos(root: Path, extensions: tuple[str, ...]) -> list[Path]:
    videos = []
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in extensions:
            videos.append(p)
    return sorted(videos)


require_ffmpeg()

In [5]:
def extract_audio_wav(video_path: Path, wav_path: Path, sample_rate: int, overwrite: bool = True) -> None:
    wav_path.parent.mkdir(parents=True, exist_ok=True)
    if wav_path.exists() and not overwrite:
        return

    cmd = [
        'ffmpeg',
        '-y' if overwrite else '-n',
        '-i',
        str(video_path),
        '-ac',
        '1',
        '-ar',
        str(sample_rate),
        '-vn',
        str(wav_path),
    ]
    run_cmd(cmd)


def detect_audio_candidates(wav_path: Path, cfg: PipelineConfig) -> np.ndarray:
    y, sr = librosa.load(str(wav_path), sr=cfg.sample_rate, mono=True)

    stft = np.abs(librosa.stft(y, n_fft=cfg.frame_length, hop_length=cfg.hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=cfg.frame_length)
    band_mask = (freqs >= cfg.low_band_hz) & (freqs <= cfg.high_band_hz)

    # Band-limited impulsive energy
    band_energy = stft[band_mask].sum(axis=0)

    # Spectral flux for sudden bursts
    spectral_flux = np.sqrt(np.sum(np.diff(stft, axis=1, prepend=stft[:, :1]) ** 2, axis=0))

    # Combined score
    score = 0.6 * robust_zscore(band_energy) + 0.4 * robust_zscore(spectral_flux)

    idx = np.where(score >= cfg.zscore_threshold)[0]
    if idx.size == 0:
        return np.array([], dtype=float)

    times = librosa.frames_to_time(idx, sr=sr, hop_length=cfg.hop_length)

    # Keep one event per minimum spacing window
    filtered = [float(times[0])]
    for t in times[1:]:
        if t - filtered[-1] >= cfg.min_event_gap_s:
            filtered.append(float(t))

    return np.array(filtered, dtype=float)

In [6]:
def _frame_stats(frame: np.ndarray) -> tuple[float, float]:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = float(gray.mean())
    return brightness, float(np.std(gray))

def visual_event_score(video_path: Path, timestamp_s: float, cfg: PipelineConfig) -> tuple[float, float]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0.0, 0.0

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    center_frame = int(timestamp_s * fps)
    radius_frames = int(cfg.visual_window_s * fps)

    start = max(0, center_frame - radius_frames)
    end = min(total_frames - 1, center_frame + radius_frames)

    brightness_vals = []
    motion_vals = []

    prev_gray = None
    for fi in range(start, end + 1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, frame = cap.read()
        if not ok:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        brightness_vals.append(float(gray.mean()))

        if prev_gray is not None:
            motion = float(np.mean(np.abs(gray.astype(np.float32) - prev_gray.astype(np.float32))))
            motion_vals.append(motion)
        prev_gray = gray

    cap.release()

    if len(brightness_vals) < 3 or len(motion_vals) < 3:
        return 0.0, 0.0

    flash_peak = float(np.max(robust_zscore(np.array(brightness_vals))))
    motion_peak = float(np.max(robust_zscore(np.array(motion_vals))))
    return flash_peak, motion_peak

def validate_events_with_video(video_path: Path, candidate_times: np.ndarray, cfg: PipelineConfig) -> list[float]:
    # Skip visual validation entirely if this is an audio-only file!
    if video_path.suffix.lower() == '.wav':
        return [float(t) for t in candidate_times]
        
    confirmed = []
    for t in candidate_times:
        flash_peak, motion_peak = visual_event_score(video_path, float(t), cfg)
        if flash_peak >= cfg.flash_zscore_threshold or motion_peak >= cfg.motion_zscore_threshold:
            confirmed.append(float(t))
    return confirmed


In [7]:
def merge_segments(segments: list[tuple[float, float]], overlap_s: float) -> list[tuple[float, float]]:
    if not segments:
        return []

    segments = sorted(segments, key=lambda x: x[0])
    merged = [segments[0]]

    for s, e in segments[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e + overlap_s:
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))

    return merged

def build_trim_segments(event_times: Iterable[float], duration_s: float, cfg: PipelineConfig) -> list[tuple[float, float]]:
    raw = []
    for t in event_times:
        s = max(0.0, float(t) - cfg.pre_event_pad_s)
        e = min(duration_s, float(t) + cfg.post_event_pad_s)
        raw.append((s, e))
    return merge_segments(raw, cfg.merge_overlap_s)

def export_segments(video_path: Path, segments: list[tuple[float, float]], export_dir: Path, overwrite: bool = True) -> list[Path]:
    export_dir.mkdir(parents=True, exist_ok=True)
    out_files: list[Path] = []
    
    is_audio = video_path.suffix.lower() == '.wav'

    for i, (start_s, end_s) in enumerate(segments, start=1):
        ext = '.wav' if is_audio else '.mp4'
        out_file = export_dir / f'{video_path.stem}_gunshot_{i:03d}{ext}'
        duration = max(0.05, end_s - start_s)

        if is_audio:
            cmd = [
                'ffmpeg',
                '-y' if overwrite else '-n',
                '-ss', f'{start_s:.3f}',
                '-i', str(video_path),
                '-t', f'{duration:.3f}',
                '-c:a', 'pcm_s16le',
                str(out_file),
            ]
        else:
            cmd = [
                'ffmpeg',
                '-y' if overwrite else '-n',
                '-ss', f'{start_s:.3f}',
                '-i', str(video_path),
                '-t', f'{duration:.3f}',
                '-c:v', 'libx264',
                '-c:a', 'aac',
                str(out_file),
            ]
        run_cmd(cmd)
        out_files.append(out_file)

    return out_files

In [8]:
def process_single_video(video_path: Path, cfg: PipelineConfig) -> dict:
    rel = video_path.relative_to(cfg.dataset_root)
    work_dir = cfg.output_root / '_work_audio' / rel.parent
    wav_path = work_dir / f'{video_path.stem}.wav'

    extract_audio_wav(video_path, wav_path, cfg.sample_rate, cfg.overwrite)
    audio_candidates = detect_audio_candidates(wav_path, cfg)

    confirmed_events = validate_events_with_video(video_path, audio_candidates, cfg)

    duration_s = get_video_duration_s(video_path)
    segments = build_trim_segments(confirmed_events, duration_s, cfg)

    export_dir = cfg.output_root / 'clips' / rel.parent
    exported = export_segments(video_path, segments, export_dir, cfg.overwrite)

    return {
        'video_path': str(video_path),
        'audio_candidates': len(audio_candidates),
        'confirmed_events': len(confirmed_events),
        'segments_exported': len(exported),
        'events_seconds': [round(x, 3) for x in confirmed_events],
        'segment_windows': [(round(s, 3), round(e, 3)) for s, e in segments],
        'exported_files': [str(p) for p in exported],
    }


def run_dataset_pipeline(cfg: PipelineConfig) -> pd.DataFrame:
    videos = discover_videos(cfg.dataset_root, cfg.video_extensions)
    if not videos:
        raise FileNotFoundError(f'No videos found in {cfg.dataset_root}.')

    rows = []
    for video in tqdm(videos, desc='Processing videos'):
        try:
            row = process_single_video(video, cfg)
            rows.append(row)
        except Exception as exc:
            LOGGER.exception('Failed for %s: %s', video, exc)
            rows.append({
                'video_path': str(video),
                'audio_candidates': 0,
                'confirmed_events': 0,
                'segments_exported': 0,
                'events_seconds': [],
                'segment_windows': [],
                'exported_files': [],
                'error': str(exc),
            })

    report_df = pd.DataFrame(rows)
    report_path = cfg.output_root / 'pipeline_report.csv'
    report_df.to_csv(report_path, index=False)
    LOGGER.info('Report saved: %s', report_path)
    return report_df

In [9]:
report = run_dataset_pipeline(CONFIG)
report[['video_path', 'audio_candidates', 'confirmed_events', 'segments_exported']].head(20)

Processing videos:   0%|          | 0/50 [00:00<?, ?it/s]

Processing videos: 100%|██████████| 50/50 [00:11<00:00,  4.29it/s]
[INFO] Report saved: c:\Desktop\Data-Cleaner\research\Data\Output\pipeline_report.csv


,video_path,audio_candidates,confirmed_events,segments_exported
0,c:\Desktop\Data-Cleaner\research\Data\SA_019A_...,3,3,1
1,c:\Desktop\Data-Cleaner\research\Data\SA_019A_...,3,3,1
2,c:\Desktop\Data-Cleaner\research\Data\SA_019A_...,3,3,1
3,c:\Desktop\Data-Cleaner\research\Data\SA_019A_...,3,3,1
4,c:\Desktop\Data-Cleaner\research\Data\SA_019A_...,3,3,1
5,c:\Desktop\Data-Cleaner\research\Data\SA_019A_...,2,2,1
6,c:\Desktop\Data-Cleaner\research\Data\SA_019B_...,3,3,1
7,c:\Desktop\Data-Cleaner\research\Data\SA_019B_...,3,3,1
8,c:\Desktop\Data-Cleaner\research\Data\SA_019B_...,3,3,1
9,c:\Desktop\Data-Cleaner\research\Data\SA_019B_...,3,3,2
